In [29]:
PATH_TO_EXPLANATION_EN = "../data/explanations/IMDB/"
PATH_TO_EVALUATION_EN = "../data/evaluation/IMDB/"

PATH_TO_TEMPLATE = "../data/evaluation/evaluation_form.txt"

MAP_LABELS_IT = {"0": "Released", "1": "NotReleased"}
MAP_LABELS_EN = {"0": "positive", "1": "negative"}

In [30]:
import glob

explanations_files = glob.glob(PATH_TO_EXPLANATION_EN + "*.json")
explanations_files

['../data/explanations/IMDB/45366.json',
 '../data/explanations/IMDB/46725.json',
 '../data/explanations/IMDB/47085.json',
 '../data/explanations/IMDB/49379.json',
 '../data/explanations/IMDB/49771.json']

In [31]:
import os
import json
from pathlib import Path

TEMPLATE_EXPLANATION = open(PATH_TO_TEMPLATE, "r").read().strip()


def process_explanation(target: Path, output_folder: Path):
    with open(target, "r") as fp:
        json_explanation = json.load(fp)
    print(json.dumps(json_explanation, indent=3))

    os.makedirs(output_folder, exist_ok=True)

    print("File:", target)
    doc_id = target.stem
    y_true = json_explanation.get('y_pred')
    y_pred = json_explanation.get('y_test')
    content = json_explanation.get('original_content')

    template = str(TEMPLATE_EXPLANATION)

    template = template.replace("@DOC_ID", str(doc_id))
    template = template.replace("@LABEL", MAP_LABELS_EN[str(y_true)])
    template = template.replace("@PREDICTION", MAP_LABELS_EN[str(y_pred)])
    template = template.replace("@TEXT", str(content).strip())

    explanation_per_hypernode = json_explanation["explanation"]
    for hyper_node_explanation_id in list(explanation_per_hypernode)[:2]:
        template_explanation = str(template)  # Copy for this specific explanation

        print("Hyper node: " + hyper_node_explanation_id)
        explanation_content = explanation_per_hypernode[hyper_node_explanation_id]

        words_l0 = explanation_content["words_l0"]
        cg_methods = explanation_content["words_cg_methods"]

        llm_search = cg_methods["llm_search"]
        semantic_search_l0 = cg_methods["semantic_search_l0"]
        semantic_search_l1 = cg_methods["semantic_search_l1"]

        explanation_object = """
### Hypernode ID
@HYPERNODE_ID

### Words highlighted from Layer 0 assigned to this hypernode in Layer 1
@WORDS_L0

### Concept grounding for hypernode in Layer 1

Method 1:
@LLM_SEARCH

Method 2:
@SEMANTIC_SEARCH_L1

Method 3:
@SEMANTIC_SEARCH_L0
        """.strip()

        explanation_object = explanation_object.replace("@HYPERNODE_ID", hyper_node_explanation_id)
        explanation_object = explanation_object.replace("@WORDS_L0", ", ".join(words_l0))

        # Method 1
        explanation_object = explanation_object.replace("@LLM_SEARCH", ", ".join(llm_search))
        # Method 2
        explanation_object = explanation_object.replace("@SEMANTIC_SEARCH_L1", ", ".join(semantic_search_l1))
        # Method 3
        explanation_object = explanation_object.replace("@SEMANTIC_SEARCH_L0", ", ".join(semantic_search_l0))

        template_explanation = template_explanation.replace("@EXPLANATION_OBJECT", explanation_object)

        output_file = Path(
            output_folder) / f"Explanation_Doc-{int(doc_id):03d}_Hypernode_{int(hyper_node_explanation_id):03d}.txt"
        with open(output_file, "w") as fp:
            fp.write(template_explanation)

    # Doc ID

    # Label
    # Prediction

    # Questions
    # 1. Is the prediction correct?


In [32]:
for explanation_file in explanations_files[:3]:
    process_explanation(Path(explanation_file), PATH_TO_EVALUATION_EN)

{
   "explanation": {
      "215": {
         "words_l0": [
            "zellweger",
            "me",
            "meara",
            "blue",
            "sensitive"
         ],
         "words_cg_methods": {
            "llm_search": [
               "actress",
               "movie",
               "drama"
            ],
            "semantic_search_l0": [
               "strang\u00e9",
               "stupek",
               "slobbered"
            ],
            "semantic_search_l1": [
               "kosciusko",
               "russkiye",
               "postels"
            ]
         }
      },
      "182": {
         "words_l0": [
            "mesmerizing",
            "funniest",
            "chronic",
            "ripe",
            "ever"
         ],
         "words_cg_methods": {
            "llm_search": [
               "best",
               "greatest",
               "classic"
            ],
            "semantic_search_l0": [
               "vanderweff",
            

In [33]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Define class distributions
majority_class_instances = 25000
minority_class_instances = 25000

# Total instances
total_instances = majority_class_instances + minority_class_instances

# Generate true labels (1 = Majority class, 0 = Minority class)
y_true = [1] * majority_class_instances + [0] * minority_class_instances

# Generate predicted labels by a majority classifier (predicts all as majority class)
y_pred = [1] * total_instances

# Calculate metrics
accuracy = accuracy_score(y_true, y_pred)
weighted_precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
weighted_recall = recall_score(y_true, y_pred, average='weighted')
weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

# Display results
print(f"Accuracy: {accuracy:.3f}")
print(f"Weighted Precision: {weighted_precision:.3f}")
print(f"Weighted Recall: {weighted_recall:.3f}")
print(f"Weighted F1-Score: {weighted_f1:.3f}")


Accuracy: 0.500
Weighted Precision: 0.250
Weighted Recall: 0.500
Weighted F1-Score: 0.333
